# AG2 (AutoGen): Multi-Agent Systems Tutorial

A tour of [AG2](https://github.com/ag2ai/ag2) — the community fork of Microsoft's AutoGen — built up from a two-agent conversation to orchestrated group chats with tools and typed output.

Covers conversable agents, specialized roles, built-in agent types, human-in-the-loop review, group-chat orchestration, tool registration, and structured outputs.

## Setup

In [ ]:
%pip install -q 'ag2[openai]<1.0' python-dotenv

In [ ]:
import json
import logging
import random
import time

from autogen import (
    ConversableAgent,
    AssistantAgent,
    UserProxyAgent,
    GroupChat,
    GroupChatManager,
)

# Shared model configuration (see config.py); reads GROQ_API_KEY from .env.
from config import llm_config

# AG2 warns about key format when using a non-OpenAI endpoint.
logging.getLogger("autogen.oai.client").setLevel(logging.ERROR)

print("AG2 modules imported successfully!")

### Model configuration

All agents share the `llm_config` from `config.py`, which points AG2 at Groq's OpenAI-compatible endpoint. Set `GROQ_API_KEY` in `.env` first.

In [ ]:
# Confirm the config loaded and the key is present.
print('model:', llm_config['config_list'][0]['model'])
print('endpoint:', llm_config['config_list'][0]['base_url'])

## Agent concepts

### Conversable agent

The base agent type: it holds a system message and can talk to other agents.

In [ ]:
from autogen import ConversableAgent

# LLM configuration (API key will be taken from environment variable)

# Student agent
student = ConversableAgent(
    name="student",
    system_message="You are a curious student. You ask clear, specific questions to learn new concepts.",
    human_input_mode="NEVER",
    llm_config=llm_config
)

# Tutor agent
tutor = ConversableAgent(
    name="tutor",
    system_message="You are a helpful tutor who provides clear and concise explanations suitable for a beginner.",
    human_input_mode="NEVER",
    llm_config=llm_config
)

# Start conversation
chat_result = student.initiate_chat(
    recipient=tutor,
    message="Can you explain what a neural network is?",
    max_turns=2,
    summary_method="reflection_with_llm"
)

print("\nFinal Summary:")
print(chat_result.summary)

`summary_method="reflection_with_llm"` asks the model to summarize the exchange at the end, rather than returning the raw transcript.

## Specialized agents

Different system messages give agents distinct roles.

In [ ]:
# Create a Technical Expert Agent
tech_expert = ConversableAgent(
    name="tech_expert",
    system_message="""You are a senior software engineer with expertise in Python, AI, and system design.
    Provide technical, detailed explanations with code examples when appropriate.
    Always consider best practices and performance implications.""",
    llm_config=llm_config,
    human_input_mode="NEVER"
)

# Create a Creative Writer Agent
creative_writer = ConversableAgent(
    name="creative_writer",
    system_message="""You are a creative writer and storyteller.
    Your responses are engaging, imaginative, and use vivid descriptions.
    You excel at making complex topics accessible through stories and analogies.""",
    llm_config=llm_config,
    human_input_mode="NEVER"
)

# Create a Business Analyst Agent
business_analyst = ConversableAgent(
    name="business_analyst",
    system_message="""You are a business analyst focused on ROI, efficiency, and strategic planning.
    Always consider business impact, costs, and practical implementation.
    Provide actionable recommendations with clear metrics.""",
    llm_config=llm_config,
    human_input_mode="NEVER"
)

agents = [tech_expert, creative_writer, business_analyst]
print("Specialized agents created!")
for agent in agents:
    print(f"- {agent.name}: {agent.system_message.split('.')[0]}.")

## Built-in agent types

`AssistantAgent` and `UserProxyAgent` cover the common assistant/executor pair.

In [ ]:
# Requirements:
# !pip install matplotlib numpy pyautogen

from autogen import AssistantAgent, UserProxyAgent
from autogen.coding import LocalCommandLineCodeExecutor

# Step 1: LLM configuration

# Step 2: Create assistant agent
assistant = AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant who writes and explains Python code clearly.",
    llm_config=llm_config
)

# Step 3: Create user proxy agent
user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=5,
    code_execution_config={
        "executor": LocalCommandLineCodeExecutor(work_dir="coding", timeout=30),
    },
)

# Step 4: Start task
chat_result = user_proxy.initiate_chat(
    recipient=assistant,
    message="Plot a sine wave using matplotlib from -2π to 2π and save the plot as sine_wave.png.",
    max_turns=4,
    summary_method="reflection_with_llm"
)

# Step 5: Display image
from IPython.display import Image, display
import os

image_path = "coding/sine_wave.png"
if os.path.exists(image_path):
    display(Image(filename=image_path))
else:
    print("Plot not found.")

# Step 6: Print summary
print("\nFinal Summary:")
print(chat_result.summary)

## Human-in-the-loop

`human_input_mode` controls when the agent pauses for a person.

### Example: a bug triage bot that checks with a human before acting.

In [ ]:
from autogen import ConversableAgent
import random

# Step 1: LLM configuration

# Step 2: Define system message
triage_system_message = """
You are a bug triage assistant. You will be given bug report summaries.

For each bug:
- If it is urgent (e.g., 'crash', 'security', or 'data loss' is mentioned), escalate it and ask the human agent for confirmation.
- If it seems minor (e.g., cosmetic, typo), suggest closing it but still ask for human review.
- Otherwise, classify it as medium priority and ask the human for review.

Once all bugs are processed, summarize what was escalated, closed, or marked as medium priority.
End by saying: "You can type exit to finish."
"""

# Step 3: Create assistant agent
triage_bot = ConversableAgent(
    name="triage_bot",
    system_message=triage_system_message,
    llm_config=llm_config
)

# Step 4: Create human agent
human = ConversableAgent(
    name="human",
    human_input_mode="ALWAYS",
)

# Step 5: Generate sample bugs
BUGS = [
    "App crashes when opening user profile.",
    "Minor UI misalignment on settings page.",
    "Password reset email not sent consistently.",
    "Typo in the About Us footer text.",
    "Database connection timeout under heavy load.",
    "Login form allows SQL injection attack.",
]

random.shuffle(BUGS)
selected_bugs = BUGS[:3]

# Step 6: Format prompt
initial_prompt = (
    "Please triage the following bug reports one by one:\n\n" +
    "\n".join([f"{i+1}. {bug}" for i, bug in enumerate(selected_bugs)])
)

# Step 7: Start conversation
human.initiate_chat(
    recipient=triage_bot,
    message=initial_prompt,
)

## Orchestration: group chats

`GroupChat` puts several agents in one conversation and `GroupChatManager` decides who speaks next, which is how AG2 handles work that needs more than two participants.

In [ ]:
from autogen import ConversableAgent, GroupChat, GroupChatManager

# LLM configuration

# System messages
planner_message = "Create a short lesson plan for 4th graders."
reviewer_message = "Review a plan and suggest up to 3 brief edits."
teacher_message = "Suggest a topic and reply DONE when satisfied."

# Create agents
lesson_planner = ConversableAgent(
    name="planner_agent",
    system_message=planner_message,
    description="Makes lesson plans.",
    llm_config=llm_config
)

lesson_reviewer = ConversableAgent(
    name="reviewer_agent",
    system_message=reviewer_message,
    description="Reviews lesson plans and suggests edits.",
    llm_config=llm_config
)

teacher = ConversableAgent(
    name="teacher_agent",
    system_message=teacher_message,
    llm_config=llm_config,
    is_termination_msg=lambda x: "DONE" in (x.get("content", "") or "").upper()
)

# Create group chat
groupchat = GroupChat(
    agents=[teacher, lesson_planner, lesson_reviewer],
    speaker_selection_method="auto"
)

# Create manager
manager = GroupChatManager(
    name="group_manager",
    groupchat=groupchat,
    llm_config=llm_config
)

# Start chat
teacher.initiate_chat(
    recipient=manager,
    message="Make a simple lesson about the moon.",
    max_turns=6,
    summary_method="reflection_with_llm"
)

## Tools

`register_function` exposes a Python function to an agent.

In [ ]:
from autogen import ConversableAgent, register_function
from typing import Annotated

# Step 1: LLM configuration

# Step 2: Define function
def is_prime(n: Annotated[int, "Positive integer"]) -> str:
    if n < 2:
        return "No"
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return "No"
    return "Yes"

# Step 3: Create agents
math_asker = ConversableAgent(
    name="math_asker",
    system_message="Ask whether a number is prime.",
    llm_config=llm_config
)

math_checker = ConversableAgent(
    name="math_checker",
    human_input_mode="NEVER",
    llm_config=llm_config
)

# Step 4: Register function
register_function(
    is_prime,
    caller=math_asker,
    executor=math_checker,
    description="Check if a number is prime. Returns Yes or No."
)

# Step 5: Start conversation
math_checker.initiate_chat(
    recipient=math_asker,
    message="Is 72 a prime number?",
    max_turns=2
)

`register_function` binds a function twice: to the agent that may *call* it (`caller`) and the agent that *executes* it (`executor`) — that split is what keeps execution controllable.

## Structured outputs

A Pydantic model as `response_format` makes replies typed rather than prose.

In [ ]:
from pydantic import BaseModel
from autogen import ConversableAgent

# Define structured output model
class TicketSummary(BaseModel):
    customer_name: str
    issue_type: str
    urgency_level: str
    recommended_action: str

# LLM configuration

# Create agent
support_agent = ConversableAgent(
    name="support_agent",
    system_message=(
        "You are a support assistant. Summarize a customer ticket using:"
        "\n- customer_name"
        "\n- issue_type (e.g. login issue, billing problem, bug report)"
        "\n- urgency_level (Low, Medium, High)"
        "\n- recommended_action"
    ),
    llm_config=llm_config
)

# Start conversation
support_agent.initiate_chat(
    recipient=support_agent,
    message="Ticket: John Doe is unable to reset his password and has an important meeting in 30 minutes.",
    max_turns=1
)

## Summary

- **ConversableAgent** — the base unit: a system message plus the ability to converse
- **GroupChat / GroupChatManager** — orchestration when more than two agents are involved
- **register_function** — tools, with calling and execution split across agents
- **human_input_mode** — where a person sits in the loop
- **response_format** — typed output instead of free text

## Author

**Anas AlGhannam**  
[github.com/AnasAlghannam](https://github.com/AnasAlghannam)